<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/58/Uber_logo_2018.svg/1024px-Uber_logo_2018.svg.png" alt="UBER LOGO" width="50%" />

# UBER Pickups 

# import libraries

In [1]:
import pandas as pd
import plotly.express as px

# Read data

In [2]:
location = pd.read_csv('c/location.csv')
print(f'location.csv : {location.shape[0]} lines / {location.shape[1]} columns')
pickup_14 = pd.read_csv('c/2014.csv')
print(f'2014.csv : {pickup_14.shape[0]} lines / {pickup_14.shape[1]} columns')
pickup = pd.read_csv('c/2015.csv')
print(f'2015.csv : {pickup.shape[0]} lines / {pickup.shape[1]} columns')

location.csv : 265 lines / 3 columns
2014.csv : 4534327 lines / 4 columns
2015.csv : 14270479 lines / 6 columns


In [3]:
a = pickup[pickup['locationid'] == 1]
a

,dispatching_base_num,pickup_date,affiliated_base_num,locationid,borough,zone
26543,b02764,2015-02-22 13:00:16,b02764,1,ewr,newark airport
125172,b02682,2015-05-17 08:15:00,b02682,1,ewr,newark airport
127618,b02764,2015-02-07 14:57:37,b02764,1,ewr,newark airport
171314,b02682,2015-05-17 11:43:00,b02682,1,ewr,newark airport
250866,b02764,2015-03-04 15:14:00,b02764,1,ewr,newark airport
...,...,...,...,...,...,...
13847889,b02765,2015-01-13 15:40:42,b02682,1,ewr,newark airport
13861832,b02764,2015-02-18 08:25:55,b02764,1,ewr,newark airport
13904130,b02764,2015-05-10 17:37:00,b02764,1,ewr,newark airport
13919543,b02765,2015-01-22 08:24:33,b02764,1,ewr,newark airport


In [4]:
b = pickup[pickup['locationid'].isin([264, 265])]
b

,dispatching_base_num,pickup_date,affiliated_base_num,locationid,borough,zone
3646,b02598,2015-01-19 10:02:46,unknown,265,unknown,unknown
10509,b02682,2015-03-28 11:39:00,b02682,265,unknown,unknown
11045,b02682,2015-03-28 11:56:00,b02682,265,unknown,unknown
11840,b02764,2015-03-17 08:46:00,b02764,265,unknown,unknown
18834,b02764,2015-02-05 05:17:29,b02764,265,unknown,unknown
...,...,...,...,...,...,...
14263004,b02765,2015-05-08 10:27:00,b02765,265,unknown,unknown
14263447,b02764,2015-05-12 20:10:00,b01197,265,unknown,unknown
14265755,b02764,2015-05-12 20:36:00,b00014,265,unknown,unknown
14266420,b02765,2015-02-20 08:39:03,b02765,265,unknown,unknown


# Identify base in 2015

In [5]:
# pickup_14['base'].isin(raw_2015['dispatching_base_num']).value_counts()

On retrouve le nombre de lignes exact de raw_2014. Aucune base ne diffère des dispatching_base_num.
Pour s'en assurer, comparons les valeurs uniques.

In [6]:
b14 = set(pickup_14['base'].unique())
b15_dispatch = set(pickup['dispatching_base_num'].unique())
b15_licence = set(pickup['affiliated_base_num'].unique())
print(len(b14), len(b15_dispatch), len(b15_licence), len(b14 & b15_dispatch & b15_licence))

5 8 285 5


Base correspond à la référence du centre de dispatch. De 2014 à 2015, le nombre des centres de dispatch a été augmenté de 3.

# Identify location pickup

locationid est présent sur uber-raw-data-janjune-15 et taxi-zone-lookup. Nous pouvons les fusionner à partir de cette colonne.

In [7]:
import geopandas as gpd
zones = gpd.read_file("taxi_zones.shp").to_crs("EPSG:4326")
pts = gpd.GeoDataFrame(df_sample, geometry=gpd.points_from_xy(df_sample["lon"], df_sample["lat"]), crs="EPSG:4326")
res = gpd.sjoin(pts, zones[["LocationID", "geometry"]], how="left", predicate="within")

ModuleNotFoundError: No module named 'geopandas'

# Treat missing values

La référence du taxi est inutile pour kmeans. On remplace par une étiquette explicite.

In [ ]:
pickup['affiliated_base_num'] = pickup['affiliated_base_num'].fillna('unknown') # drop lines where values are missing. 
missing_values = pickup.isnull().sum()/pickup.shape[0]*100
print(missing_values)

# Analyze raw_2014

Les fichiers tracent 6 mois d'exploitation en 2014 d'avril à septembre. Les informations diffèrent : date/time, lat, lon, base.
Cette fois, nous disposons des coordonnées des pickups.
Nous devons identifier les emplacements et établir si base est la référence du dispatch ou du taxi. 

In [ ]:
pickup_14_volume = pickup_14["base"].value_counts()
pickup_volume = pickup["dispatching_base_num"].value_counts()
print(pickup_14_volume, pickup_volume)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

v14 = pickup_14["base"].value_counts().rename("2014")
v15 = pickup["dispatching_base_num"].value_counts().rename("2015")

pd.concat([v14, v15], axis=1).plot(kind="bar", figsize=(8, 5))
plt.xlabel("dispatch_base")
plt.ylabel("pickups")
plt.show()

In [ ]:
import pandas as pd
import plotly.express as px

v = pd.concat([pickup_14["base"].value_counts().rename("2014"),
               pickup["dispatching_base_num"].value_counts().rename("2015")], axis=1)

fig = px.bar(v.reset_index(), x="index", y=["2014", "2015"], barmode="group",
             title="Nombre de pickups par base de dispatch",
             labels={"index": "Base", "value": "Pickups", "variable": "Year"})
fig.show()

# Analyze one hour on one day

In [ ]:
pickup.head(1)

In [ ]:
pickup_14['date/time'].dtype

In [ ]:
pickup_14['pickup_date'].dtype

In [ ]:
begin = pd.Timestamp('2014-04-01 00:00:00')
end = begin + pd.Timedelta(hours=1)

t1 = pickup_14[(pickup_14['date/time'] >= begin) & (pickup_14['date/time'] < end)]

In [ ]:
t1

In [ ]:
pickup_15 = t1['locationid'].unique()
# pickup_15 = t1.groupby('borough')['location_id'].value_counts()
print(pickup_15)

In [ ]:
bases_median = t1.groupby('base')[['lat', 'lon']].median().reset_index()

In [ ]:
print(bases_median)

In [ ]:
fig = px.scatter_map(bases_median, lat="lat", lon="lon", color="base", zoom=10, map_style="carto-positron")
fig.show()

Les bases b02617 et b02764 se superposent. Let's record bases and search for duplicated values.

# Create pickup

Définir les zones
Vérifier le nombre de prises en charge par zone / location/id
définir statut

Voir les pickups de 2014 sur la carte https://app.jedha.co/course/http-and-apis-ft/api-practical-demo-ft


In [ ]:
bases = pickup_14.groupby('base')[['lat', 'lon']].median().reset_index()

In [ ]:
pickup_14[['base', 'lon', 'lat']].to_csv("data/bases.csv", index=False)

In [ ]:
print(bases.head(1))

# Search for duplicated values

In [ ]:
duplicated_values = bases[bases.duplicated(subset=["lat", "lon"], keep=False)]
print(duplicated_values.sort_values(['base', "lat", "lon"]))

In [ ]:
with pd.option_context("display.precision", 15):
    print(bases_median)

There are no duplicated values, only values very similar. Des points proches signalent justement une forte densité de pickups, donc une hot-zone. Les supprimer ferait perdre cette information et fausserait les clusters (KMeans comme DBSCAN).

In [ ]:
# bases_t1 = bases_median.round({"lat": 4, "lon": 4})
# print(bases_t1[bases_t1.duplicated(subset=["lat", "lon"], keep=False)])

Vérifier l'utilité de locationid.

# Detect outlyers

On cherche seulement les points aberrants. Dans l'eau ou en dehors de l'état de new york. Sur sample, il n'y en a pas.

# Create sample

In [ ]:
mask = pickup[pickup['dispatching_base_num'].isin(bases_median['base'])]
n = 5000  # taille cible, choix provisoire 5000 puis 2500
frac = n / len(mask)
sample = mask.groupby('dispatching_base_num').sample(frac=frac, random_state=42)
print(frac, len(sample))

In [ ]:
sample.to_csv("data/sample.csv", index=False)

rééchantillonnage sur locationid